In [ ]:
import gc

try:
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception:
    gc.collect()
    print('Torch not available, skip GPU cache clear.')


# 第 1 课：RAG 入门

这一课我们先不急着接外部 API，也不急着上向量数据库。

先把 RAG 最核心的主线彻底搞明白：

- 什么是 RAG
- 为什么大模型需要 RAG
- 一个最小可运行的 RAG 流程长什么样

你可以把这节课理解成 RAG 版的“Q-learning 第一课”。先把骨架搭起来，后面再逐步升级。


## 1. 什么是 RAG

`RAG` = `Retrieval-Augmented Generation`

中文通常叫：

**检索增强生成**

它的核心思想非常简单：

1. 用户提出问题
2. 系统先去资料库里找相关内容
3. 再把找到的内容连同问题一起交给大模型回答

也就是说，模型不是只靠自己参数里的知识，而是：

**先查资料，再回答。**


## 2. 为什么需要 RAG

如果你只用一个裸模型，通常会遇到这些问题：

- 模型知识可能过时
- 模型不知道你的私有资料
- 模型可能幻觉，胡编一个看起来很像真的答案
- 模型没法直接引用你指定的文档内容

RAG 的作用就是缓解这些问题：

- 让回答基于外部资料
- 让模型能用企业内部知识库
- 提高可控性和可解释性
- 更容易附带出处和证据


## 3. 一个最小 RAG 流程

最小版 RAG 可以拆成三步：

1. `Indexing`
   把资料整理成可检索的片段。

2. `Retrieval`
   当用户提问时，从资料里找最相关的片段。

3. `Generation`
   把“问题 + 检索结果”拼成 prompt，再让模型回答。

你后面学的所有高级 RAG，基本都还是围绕这三步在升级。


In [ ]:
from collections import Counter
import math
import re

documents = [
    {
        'id': 'doc_1',
        'title': 'RAG Overview',
        'text': 'RAG means retrieval-augmented generation. The system first retrieves relevant knowledge, then uses that context to help the language model answer questions.'
    },
    {
        'id': 'doc_2',
        'title': 'Embeddings',
        'text': 'Embeddings convert text into vectors. Similar texts should have similar vectors, which helps semantic retrieval.'
    },
    {
        'id': 'doc_3',
        'title': 'Chunking',
        'text': 'Chunking splits long documents into smaller passages. Good chunking often improves retrieval quality because the returned context is more focused.'
    },
    {
        'id': 'doc_4',
        'title': 'Evaluation',
        'text': 'RAG evaluation often checks retrieval quality, answer correctness, groundedness, and whether the response actually uses the retrieved evidence.'
    }
]

print('文档数量:', len(documents))
for d in documents:
    print('-', d['id'], d['title'])


## 4. 我们先做一个最简单的“索引”

真正的生产级 RAG 会用：
- embedding 模型
- 向量数据库
- 稀疏检索 / 稠密检索 / 混合检索

但第一课先不把系统堆太复杂。

我们先做一个最小可运行版本：

- 用词频构造一个非常简化的 bag-of-words 表示
- 用余弦相似度做检索

虽然它不如 embedding 高级，但非常适合先把检索流程看懂。


In [ ]:
def tokenize(text):
    return re.findall(r"[a-zA-Z]+", text.lower())


vocab = sorted(set(token for doc in documents for token in tokenize(doc['text'])))
vocab_index = {token: i for i, token in enumerate(vocab)}

print('词表大小:', len(vocab))
print('前 20 个 token:', vocab[:20])


In [ ]:
def text_to_vector(text, vocab_index):
    vec = [0.0] * len(vocab_index)
    counts = Counter(tokenize(text))
    for token, count in counts.items():
        if token in vocab_index:
            vec[vocab_index[token]] = float(count)
    return vec


def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


doc_vectors = {
    doc['id']: text_to_vector(doc['text'], vocab_index)
    for doc in documents
}

print('已完成最小索引构建。')


## 5. Retrieval：怎么找最相关的文档

现在用户一提问，我们就：

1. 把 query 也转成向量
2. 和每篇文档算相似度
3. 取分数最高的前 k 篇

这就是最小版的检索器。


In [ ]:
def retrieve(query, documents, doc_vectors, vocab_index, top_k=2):
    query_vec = text_to_vector(query, vocab_index)
    scored = []
    for doc in documents:
        score = cosine_similarity(query_vec, doc_vectors[doc['id']])
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]


query = 'How does RAG use retrieved evidence to answer questions?'
results = retrieve(query, documents, doc_vectors, vocab_index, top_k=2)

print('query:', query)
print()
for score, doc in results:
    print(f"score={score:.3f} | {doc['id']} | {doc['title']}")
    print(doc['text'])
    print()


## 6. Generation：怎么把检索结果交给模型

真正的 RAG 里，这一步会把检索到的内容拼成 prompt，例如：

- system prompt
- 用户问题
- 检索到的文档片段

然后交给 LLM 回答。

这节课我们先不调用真实模型，先把 prompt 结构看清楚。


In [ ]:
def build_rag_prompt(query, retrieved_docs):
    context_blocks = []
    for i, (_, doc) in enumerate(retrieved_docs, start=1):
        context_blocks.append(f"[Document {i}] {doc['title']}\n{doc['text']}")

    context = "\n\n".join(context_blocks)
    prompt = f"""You are a helpful assistant. Answer the question using the retrieved context.

Question:
{query}

Retrieved Context:
{context}

Answer:
"""
    return prompt


rag_prompt = build_rag_prompt(query, results)
print(rag_prompt)


## 7. 一个最小 RAG 系统，完整串起来

下面我们把最小版 RAG 写成一个完整函数：

- 输入用户问题
- 先检索
- 再构造 prompt

在真实项目里，你只需要把最后的“伪回答器”换成真正的大模型调用即可。


In [ ]:
def toy_generator(query, retrieved_docs):
    # 这里故意用一个非常简单的假生成器，只为了让你看清 RAG 管线。
    joined = ' '.join(doc['text'] for _, doc in retrieved_docs)
    return f"Toy answer based on retrieved context: {joined}"


def toy_rag_pipeline(query, documents, doc_vectors, vocab_index, top_k=2):
    retrieved = retrieve(query, documents, doc_vectors, vocab_index, top_k=top_k)
    prompt = build_rag_prompt(query, retrieved)
    answer = toy_generator(query, retrieved)
    return retrieved, prompt, answer


query2 = 'Why is chunking important in RAG?'
retrieved2, prompt2, answer2 = toy_rag_pipeline(query2, documents, doc_vectors, vocab_index)

print('检索结果:')
for score, doc in retrieved2:
    print(f"score={score:.3f} | {doc['title']}")

print('\n生成答案:')
print(answer2)


## 8. 这还不是真正工业级 RAG

这节课故意做得很简单，所以你要知道它和真实 RAG 的差距：

- 我们现在用的是词频，不是 embedding
- 我们现在没有 chunking 流程
- 我们现在没有 rerank
- 我们现在没有真实 LLM 生成
- 我们现在还没有 evaluation

但没关系，这节课最重要的是：

**你已经把 RAG 的骨架跑通了。**


## 9. RAG Evaluation 先有个最小概念

你刚刚提到想学 `RAG + Evaluation`，所以这里先给你埋一个最小概念。

RAG 的评估通常至少分两层：

1. `Retrieval quality`
   检索有没有把真正相关的文档找出来。

2. `Generation quality`
   回答是不是正确、是不是基于检索内容、有没有幻觉。

也就是说，RAG 不是只评估“最后答案好不好”，还要评估“检索这一步做得对不对”。


In [ ]:
gold_relevant = {'doc_3'}
retrieved_ids = {doc['id'] for _, doc in retrieved2}

hit = len(gold_relevant & retrieved_ids) > 0
precision_at_k = len(gold_relevant & retrieved_ids) / len(retrieved_ids)
recall_at_k = len(gold_relevant & retrieved_ids) / len(gold_relevant)

print('最小 evaluation 示例:')
print('hit@k =', hit)
print('precision@k =', round(precision_at_k, 3))
print('recall@k =', round(recall_at_k, 3))


## 10. 这节课最该记住什么

如果你想抓住这节课最核心的主线，就记住：

- RAG 不是一个神秘大黑盒
- 它就是“先检索，再生成”
- 最小骨架永远是：`index -> retrieve -> generate`

你后面学的 embedding、向量库、rerank、hybrid search、eval，都是在给这条主线做升级。


## 11. 下一课最自然学什么

学完这一课后，最自然的下一步就是：

**文本切块与 chunking 策略**

因为真实 RAG 的第一步，往往不是“直接拿整篇文档检索”，而是先把文档切成合适的片段。
